In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
try:
    from google.colab import drive
    IS_COLAB_DRIVE = True
except Exception:
    IS_COLAB_DRIVE = False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: Install dependencies
# ⚠️ KHÔNG upgrade torch/torchvision trên Kaggle — wheel PyPI có thể không build
# cho GPU arch đang dùng (T4/P100/L4) -> CUDA error "no kernel image for device".
# Chỉ upgrade các lib phụ, giữ nguyên torch của Kaggle.
%pip install -q --upgrade transformers accelerate huggingface_hub spacy decord
!python -m spacy download en_core_web_sm -q

import torch
print('✅ deps ready | torch', torch.__version__, '| cuda', torch.version.cuda,
      '| device_cap', torch.cuda.get_device_capability() if torch.cuda.is_available() else None,
      '| gpu', torch.cuda.get_device_name() if torch.cuda.is_available() else 'cpu')


In [ ]:
# CELL 1B: Install Detectron2 (Faster R-CNN) - robust install
# NOTE: chạy cell này trước CELL load model
import os
import sys
import subprocess
from pathlib import Path

D2_DIR = Path('/content/detectron2')

print('=== Detectron2 setup ===')
print('Python:', sys.version.split()[0])

# 1) Clone source nếu chưa có
if not D2_DIR.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/facebookresearch/detectron2.git',
        str(D2_DIR)
    ], check=True)
else:
    print(f'Using existing source: {D2_DIR}')

# 2) Install deps
subprocess.run([sys.executable, '-m', 'pip', 'install', 'cython', 'pyyaml', 'tqdm', 'opencv-python', 'imageio', 'decord'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pycocotools'], check=True)

# 3) Editable install from detectron2 source dir
os.chdir(D2_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# 4) Keep working in detectron2 dir + verify import
# (tránh shadow import khi cwd là /content)
d2_src = str(D2_DIR)
if d2_src not in sys.path:
    sys.path.insert(0, d2_src)

import detectron2
from detectron2.config import get_cfg
print('✅ detectron2 import OK')
print('cwd:', Path.cwd())
print('detectron2 module:', detectron2.__file__)

In [ ]:
# CELL 2: Imports & config
import os, json, pickle, zipfile, time, re
from pathlib import Path
from collections import OrderedDict

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torchvision.ops import nms
from tqdm.auto import tqdm
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    AutoModel,
    AutoTokenizer,
 )
from huggingface_hub import HfApi, login, hf_hub_download
import spacy


def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


# ============================================
# 🔑 PART CONFIG — chỉ đổi PART_ID (1..5) cho mỗi lần chạy
# ============================================
PART_ID = 1  # <-- ĐỔI 1..5 cho mỗi notebook run
PART_NAME = f'causal_vidqa_raw_videos_part_{PART_ID:02d}_of_5'
PART_ZIP  = f'{PART_NAME}.zip'

class Config:
    # === Inputs ===
    QA_ROOT       = '/content/data/text-annotation/QA'  # text.json/answer.json per video
    VIDEO_ROOT    = '/content/data/raw-casual'
    SPLIT_LIST    = None

    # === HF input dataset (5 part zip) ===
    HF_VIDEO_REPO = 'DanielQ07/causal-vidqa-raw-video-full-5zip'
    HF_VIDEO_FILE = PART_ZIP

    # === Models ===
    GDINO_NAME = 'IDEA-Research/grounding-dino-base'
    DEBERTA_NAME = 'microsoft/deberta-v3-base'
    FRCNN_CFG = 'COCO-Detection/faster_rcnn_R_101_C4_3x.yaml'
    FRCNN_SCORE_THRESH = 0.05

    # === Filtering ===
    N_FRAMES = 16
    BOX_THRESH = 0.20
    TEXT_THRESH = 0.25
    FOCUS_THRESH_BONUS = 0.10
    NMS_IOU = 0.5
    MIN_AREA_RATIO = 0.0005
    MAX_BOXES = 12
    MAX_PROMPT_NOUNS = 20

    # === GDINO size cap (set None/0 to disable resizing) ===
    GDINO_MAX_SIDE = None
    GDINO_FRAME_CHUNK = 4

    # === FasterRCNN ROI feature ===
    HIDDEN_DIM = 2048

    # === Throughput knobs ===
    VIDEO_BATCH = 1
    DEBERTA_BATCH = 256
    USE_DECORD = True
    PREFETCH_WORKERS = 4
    EMPTY_CACHE_EVERY = 30

    # === Output ===
    OUTPUT_DIR = Path(f'/content/out_{PART_NAME}')
    ZIP_NAME = PART_ZIP  # output zip name = part name

    # === HF output ===
    HF_TOKEN = read_secret('HF_TOKEN')  # Colab Secret/environment; never hard-code tokens.
    HF_REPO_ID = 'DanielQ07/kltn'
    HF_OUT_PATH = f'gdinofasterrcnn/{PART_ZIP}'

    # === Misc ===
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
    NOTEBOOK_ID = 0
    TOTAL_NODES = 1

cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'🎯 PART_ID={PART_ID} | PART_NAME={PART_NAME}')
print(f'   In:  HF dataset {cfg.HF_VIDEO_REPO} / {cfg.HF_VIDEO_FILE}')
print(f'   Out: HF dataset {cfg.HF_REPO_ID} / {cfg.HF_OUT_PATH}')
print(f'Device: {cfg.DEVICE} | dtype: {cfg.DTYPE}'
      f' | video_batch: {cfg.VIDEO_BATCH} | gdino_chunk: {cfg.GDINO_FRAME_CHUNK}'
      f' | prefetch: {cfg.PREFETCH_WORKERS}')

In [ ]:
# CELL 3: Download data — videos từ HF (theo PART_ID), annotations từ Kaggle
import os, shutil, subprocess, zipfile as _zipfile
from pathlib import Path

IS_COLAB = 'COLAB_GPU' in os.environ or Path('/content').exists()

DATA_ROOT = Path('/content/data')
VIDEO_DIR = DATA_ROOT / 'raw-casual'
ANNO_DIR = DATA_ROOT / 'text-annotation'

# ============================================
# 1. ANNOTATIONS từ Kaggle (giữ nguyên - dùng cho mọi part)
# ============================================
def _setup_kaggle_creds():
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        if username and key:
            os.environ['KAGGLE_USERNAME'] = username
            os.environ['KAGGLE_KEY'] = key
            print('✅ Kaggle credentials loaded from Colab Secrets')
            return True
    except Exception as e:
        print(f'⚠️ Could not load Colab Secrets: {e}')
    return False


def _kaggle_download(slug, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    if any(target_dir.iterdir()):
        print(f'⏭  {slug}: đã có data — skip')
        return
    print(f'⬇️  Downloading {slug} → {target_dir} ...')
    cmd = ['kaggle', 'datasets', 'download', '-d', slug,
           '-p', str(target_dir), '--unzip', '--quiet']
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print('STDERR:', res.stderr)
        raise RuntimeError(f'Kaggle download fail cho {slug}')
    print(f'   ✅ {slug} done')


if IS_COLAB:
    try:
        subprocess.check_output(['kaggle', '--version'])
    except Exception:
        subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    _setup_kaggle_creds()
    _kaggle_download('lusnaw/text-annotation', ANNO_DIR)

# ============================================
# 2. VIDEOS từ HuggingFace (1 part zip)
# ============================================
print(f'\n⬇️  Downloading video part từ HF: {cfg.HF_VIDEO_REPO}/{cfg.HF_VIDEO_FILE}')
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# Login HF nếu có token
if cfg.HF_TOKEN:
    try:
        login(token=cfg.HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print(f'⚠️ HF login fail: {e}')

# Download zip về cache local rồi extract
zip_local = hf_hub_download(
    repo_id=cfg.HF_VIDEO_REPO,
    filename=cfg.HF_VIDEO_FILE,
    repo_type='dataset',
    local_dir='/content/_hf_cache',
)
print(f'   ✅ Downloaded: {zip_local}')

# Extract chỉ khi chưa có (skip nếu VIDEO_DIR đã có mp4)
existing_mp4 = list(VIDEO_DIR.rglob('*.mp4'))
if len(existing_mp4) >= 100:
    print(f'⏭  {len(existing_mp4)} mp4 đã có trong {VIDEO_DIR} — skip extract')
else:
    print(f'📦 Extracting {zip_local} → {VIDEO_DIR} ...')
    with _zipfile.ZipFile(zip_local, 'r') as zf:
        zf.extractall(VIDEO_DIR)
    n_mp4 = len(list(VIDEO_DIR.rglob('*.mp4')))
    print(f'   ✅ Extracted: {n_mp4} mp4')

# ============================================
# 3. Override paths trong cfg
# ============================================
cfg.QA_ROOT = str(ANNO_DIR / 'QA')
cfg.VIDEO_ROOT = str(VIDEO_DIR)

print('\n🔄 cfg paths:')
for k in ('QA_ROOT', 'VIDEO_ROOT', 'OUTPUT_DIR'):
    v = getattr(cfg, k)
    flag = '✅' if Path(v).exists() else '❌'
    print(f'   {flag} {k}: {v}')

qa_n = len(list(Path(cfg.QA_ROOT).iterdir())) if Path(cfg.QA_ROOT).exists() else 0
vid_n = len(list(Path(cfg.VIDEO_ROOT).rglob('*.mp4'))) if Path(cfg.VIDEO_ROOT).exists() else 0
print(f'\n📊 QA dirs: {qa_n} | mp4 (part {PART_ID}): {vid_n}')

In [ ]:
# CELL 3: Load models (fp16) & spaCy
# Ensure detectron2 source path + cwd are correct before imports
import os
import sys
from pathlib import Path

D2_DIR = Path('/content/detectron2')
os.chdir(D2_DIR)
if str(D2_DIR) not in sys.path:
    sys.path.insert(0, str(D2_DIR))

from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2 import model_zoo

print('Loading GroundingDINO...')
gdino_processor = AutoProcessor.from_pretrained(cfg.GDINO_NAME)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    cfg.GDINO_NAME, torch_dtype=cfg.DTYPE
).to(cfg.DEVICE).eval()

print('Loading FasterRCNN R101-C4...')
frcnn_cfg = get_cfg()
frcnn_cfg.merge_from_file(model_zoo.get_config_file(cfg.FRCNN_CFG))
frcnn_cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(cfg.FRCNN_CFG)
frcnn_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = cfg.FRCNN_SCORE_THRESH
frcnn_cfg.MODEL.DEVICE = cfg.DEVICE
frcnn_model = build_model(frcnn_cfg)
DetectionCheckpointer(frcnn_model).load(frcnn_cfg.MODEL.WEIGHTS)
frcnn_model.eval()

print('Loading DeBERTa-v3-base...')
deberta_tokenizer = AutoTokenizer.from_pretrained(cfg.DEBERTA_NAME)
deberta_model = AutoModel.from_pretrained(
    cfg.DEBERTA_NAME, torch_dtype=cfg.DTYPE
).to(cfg.DEVICE).eval()

print('Loading spaCy en_core_web_sm...')
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

for m in (gdino_model, frcnn_model, deberta_model):
    for p in m.parameters():
        p.requires_grad_(False)
print('✅ Models ready')

In [ ]:
# CELL 4: Prompt builder (per-video QA nouns via spaCy)
# Bổ sung: trả thêm focus_nouns = các từ nằm trong [...] (vd [person_1] -> person).
PLACEHOLDER_RE = re.compile(r'\[([a-zA-Z]+)_\d+\]')  # e.g. [person_1] -> person
STOP_NOUNS = {
    'thing', 'object', 'something', 'stuff', 'someone', 'one', 'way', 'time', 'kind', 'sort',
    'lot', 'piece', 'side', 'end', 'part', 'place', 'people',
    'left', 'right', 'top', 'bottom', 'front', 'back',
}


def _qa_text_for_video(qa_dir):
    text = json.load(open(qa_dir / 'text.json', encoding='utf-8'))
    paragraphs = []
    for block in text.values():
        if not isinstance(block, dict):
            continue
        if 'question' in block:
            paragraphs.append(block['question'])
        for k in ('answer', 'reason'):
            if k in block and isinstance(block[k], list):
                paragraphs.extend(block[k])
    return ' '.join(paragraphs)


def _clean_token(tok):
    val = tok.lemma_.strip().lower()
    if not val or len(val) < 2:
        return None
    if not re.fullmatch(r'[a-zA-Z]+', val):
        return None
    return val


def _collect_noun_phrases(doc):
    phrases = []
    i = 0
    while i < len(doc):
        tok = doc[i]
        if tok.pos_ not in ('NOUN', 'PROPN'):
            i += 1
            continue
        j = i
        chunk = []
        while j < len(doc) and doc[j].pos_ in ('NOUN', 'PROPN'):
            t = _clean_token(doc[j])
            if t:
                chunk.append(t)
            j += 1
        if chunk:
            phrases.append(' '.join(chunk))
        i = j
    return phrases


def build_prompt_for_video(video_id):
    """Return (prompt_str, ordered_nouns, focus_nouns).

    focus_nouns: các lemma trong [...] của text.json — đây là object chính cần
    phải detect cho bằng được nên sẽ dùng threshold thấp hơn.
    """
    qa_dir = Path(cfg.QA_ROOT) / video_id
    if not qa_dir.exists():
        return None, [], []
    raw = _qa_text_for_video(qa_dir)
    # Lấy focus nouns TRƯỚC khi unwrap brackets
    focus_raw = [m.group(1).lower() for m in PLACEHOLDER_RE.finditer(raw)]
    focus_nouns = []
    seen_focus = set()
    for f in focus_raw:
        if f in STOP_NOUNS or f in seen_focus or len(f) < 2 or not f.isalpha():
            continue
        seen_focus.add(f)
        focus_nouns.append(f)

    # collapse [person_1] -> person, [chair_2] -> chair
    raw = PLACEHOLDER_RE.sub(lambda m: m.group(1).lower(), raw)
    if nlp is None:
        raise RuntimeError('spaCy chưa load. Hãy chạy Cell 3 trước.')
    doc = nlp(raw.lower())
    nouns = []
    seen = set()
    # Đưa focus_nouns lên đầu prompt để ưu tiên
    for f in focus_nouns:
        seen.add(f); nouns.append(f)
    for phrase in _collect_noun_phrases(doc):
        words = [w for w in phrase.split() if w not in STOP_NOUNS]
        if not words:
            continue
        phrase = ' '.join(words)
        if phrase in seen:
            continue
        seen.add(phrase)
        nouns.append(phrase)
        if len(nouns) >= cfg.MAX_PROMPT_NOUNS:
            break
    if not nouns:
        nouns = ['person', 'object']
    prompt = ' . '.join(nouns) + ' .'
    return prompt, nouns, focus_nouns


def _ensure_nlp_ready():
    global nlp
    if nlp is None:
        nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])


_ensure_nlp_ready()
sample_vid = next(iter(os.listdir(cfg.QA_ROOT)))
p, n, f = build_prompt_for_video(sample_vid)
print(f'[{sample_vid}] {len(n)} nouns | focus={f}')
print('  prompt:', p[:160], '...')

In [ ]:
# CELL 5: Frame sampling + GroundingDINO inference (batched)

# --- decord (CPU) decode, fallback cv2 nếu lỗi ---
_DECORD_OK = False
if cfg.USE_DECORD:
    try:
        import decord
        decord.bridge.set_bridge('native')
        _DECORD_OK = True
        print('✅ decord ready (CPU bridge)')
    except Exception as e:
        print('⚠️  decord không dùng được, fallback cv2:', e)

import warnings
warnings.filterwarnings(
    'ignore',
    message=r'.*The key `labels`.*Use `text_labels` instead.*',
    category=FutureWarning,
    module=r'transformers\.models\.grounding_dino\.processing_grounding_dino',
)


def _sample_frames_cv2(video_path, n_frames):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None, None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    indices = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame_bgr = cap.read()
        if not ok or frame_bgr is None:
            frames.append(None)
        else:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, indices.tolist(), total


def _sample_frames_decord(video_path, n_frames):
    try:
        vr = decord.VideoReader(str(video_path), num_threads=1)
    except Exception:
        return _sample_frames_cv2(video_path, n_frames)
    total = len(vr)
    if total == 0:
        return None, None, 0
    indices = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    try:
        batch = vr.get_batch(indices.tolist()).asnumpy()  # (T, H, W, 3) RGB
    except Exception:
        return _sample_frames_cv2(video_path, n_frames)
    frames = [batch[i] for i in range(batch.shape[0])]
    return frames, indices.tolist(), int(total)


def sample_frames(video_path, n_frames):
    if _DECORD_OK:
        return _sample_frames_decord(video_path, n_frames)
    return _sample_frames_cv2(video_path, n_frames)


def _resize_for_gdino(img):
    max_side = getattr(cfg, 'GDINO_MAX_SIDE', None)
    if not max_side or max_side <= 0:
        return img
    h, w = img.shape[:2]
    long_side = max(h, w)
    if long_side <= max_side:
        return img
    scale = max_side / float(long_side)
    nh = max(1, int(round(h * scale)))
    nw = max(1, int(round(w * scale)))
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)


def _gdino_inputs(images, prompts):
    """Wrapper gọi gdino_processor với padding + truncation cho text.

    Xử lý case processor cũ không hỗ trợ kwargs 'padding'/'truncation' trực tiếp
    (fallback sang gọi qua processor.tokenizer nếu cần).
    """
    try:
        return gdino_processor(
            images=images, text=prompts,
            padding=True, truncation=True, return_tensors='pt',
        )
    except TypeError:
        # Một số version gdino_processor forward kwargs xuống image proc → lỗi.
        # Tokenize thủ công rồi ghép với image features.
        tok = gdino_processor.tokenizer(
            prompts, padding=True, truncation=True, return_tensors='pt',
        )
        img = gdino_processor.image_processor(images=images, return_tensors='pt')
        out = {**img, **tok}
        return out


@torch.no_grad()
def run_gdino_batch(frames_rgb, prompt, box_thresh=None, text_thresh=None):
    """GroundingDINO trên 1 video: list frame + 1 prompt (giữ nguyên cho smoke test)."""
    bt = cfg.BOX_THRESH if box_thresh is None else box_thresh
    tt = cfg.TEXT_THRESH if text_thresh is None else text_thresh

    valid_idx = [i for i, f in enumerate(frames_rgb) if f is not None]
    results = [None] * len(frames_rgb)
    if not valid_idx:
        return results
    images = [frames_rgb[i] for i in valid_idx]
    target_sizes = [(im.shape[0], im.shape[1]) for im in images]
    images = [_resize_for_gdino(im) for im in images]
    chunk = getattr(cfg, 'GDINO_FRAME_CHUNK', None)
    if not chunk or chunk <= 0:
        chunk = len(images)
    for s in range(0, len(images), chunk):
        sub_images = images[s:s + chunk]
        sub_sizes = target_sizes[s:s + chunk]
        inputs = _gdino_inputs(sub_images, [prompt] * len(sub_images))
        inputs = {k: v.to(cfg.DEVICE) for k, v in inputs.items()}
        with torch.autocast(device_type='cuda', dtype=cfg.DTYPE, enabled=cfg.DEVICE == 'cuda'):
            outputs = gdino_model(**inputs)
        try:
            post = gdino_processor.post_process_grounded_object_detection(
                outputs, inputs['input_ids'],
                box_threshold=bt, text_threshold=tt,
                target_sizes=sub_sizes,
            )
        except TypeError:
            post = gdino_processor.post_process_grounded_object_detection(
                outputs, threshold=bt, text_threshold=tt,
                target_sizes=sub_sizes,
            )
        for j, res in enumerate(post):
            results[valid_idx[s + j]] = res
    return results


@torch.no_grad()
def run_gdino_multi(video_items, box_thresh=None, text_thresh=None):
    """GroundingDINO cho nhiều video cùng lúc.

    video_items: list[(vid, frames_rgb, prompt)] — frames_rgb có thể chứa None.
    Trả về dict {vid: list_of_frame_results} cùng độ dài frames_rgb.
    """
    bt = cfg.BOX_THRESH if box_thresh is None else box_thresh
    tt = cfg.TEXT_THRESH if text_thresh is None else text_thresh

    flat_images, flat_prompts, flat_target = [], [], []
    flat_meta = []  # (vid, slot)
    out = {vid: [None] * len(frames) for vid, frames, _ in video_items}
    for vid, frames, prompt in video_items:
        for slot, fr in enumerate(frames):
            if fr is None:
                continue
            orig_h, orig_w = fr.shape[:2]
            flat_images.append(_resize_for_gdino(fr))
            flat_prompts.append(prompt)
            flat_target.append((orig_h, orig_w))
            flat_meta.append((vid, slot))
    if not flat_images:
        return out

    chunk = getattr(cfg, 'GDINO_FRAME_CHUNK', None)
    if not chunk or chunk <= 0:
        chunk = len(flat_images)
    all_posts = []
    for s in range(0, len(flat_images), chunk):
        sub_images = flat_images[s:s + chunk]
        sub_prompts = flat_prompts[s:s + chunk]
        sub_targets = flat_target[s:s + chunk]
        inputs = _gdino_inputs(sub_images, sub_prompts)
        inputs = {k: v.to(cfg.DEVICE) for k, v in inputs.items()}
        with torch.autocast(device_type='cuda', dtype=cfg.DTYPE, enabled=cfg.DEVICE == 'cuda'):
            outputs = gdino_model(**inputs)
        try:
            post = gdino_processor.post_process_grounded_object_detection(
                outputs, inputs['input_ids'],
                box_threshold=bt, text_threshold=tt,
                target_sizes=sub_targets,
            )
        except TypeError:
            post = gdino_processor.post_process_grounded_object_detection(
                outputs, threshold=bt, text_threshold=tt,
                target_sizes=sub_targets,
            )
        all_posts.extend(post)
    for (vid, slot), res in zip(flat_meta, all_posts):
        out[vid][slot] = res
    return out


def filter_boxes(res, h, w, allowed_nouns, focus_nouns=()):
    """Apply area filter + NMS + max-box cap; clean labels; two-tier threshold."""
    if res is None or len(res['boxes']) == 0:
        return (np.zeros((0, 4), dtype=np.float32),
                np.zeros((0,), dtype=np.float32),
                [])
    boxes = res['boxes'].detach().cpu().float()
    scores = res['scores'].detach().cpu().float()
    raw_labels = res.get('text_labels', res.get('labels', []))

    # area filter
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    min_area = cfg.MIN_AREA_RATIO * h * w
    keep_area = areas >= min_area
    if keep_area.sum() == 0:
        return (np.zeros((0, 4), dtype=np.float32),
                np.zeros((0,), dtype=np.float32), [])
    boxes = boxes[keep_area]
    scores = scores[keep_area]
    raw_labels = [raw_labels[i] for i, k in enumerate(keep_area.tolist()) if k]

    # NMS
    keep = nms(boxes, scores, cfg.NMS_IOU)
    boxes = boxes[keep]
    scores = scores[keep]
    raw_labels = [raw_labels[i] for i in keep.tolist()]

    # clean labels
    allowed_set = set(allowed_nouns)
    focus_set = set(focus_nouns)
    cleaned = []
    for lab in raw_labels:
        words = re.findall(r'[a-zA-Z]+', str(lab).lower())
        match = next((w for w in words if w in allowed_set), None)
        if match is None and words:
            match = words[0]
        cleaned.append(match or 'object')

    # two-tier threshold
    bonus = cfg.FOCUS_THRESH_BONUS
    strict_thresh = cfg.BOX_THRESH
    focus_thresh = max(0.05, cfg.BOX_THRESH - bonus)
    keep_mask = []
    for sc, lab in zip(scores.tolist(), cleaned):
        if lab in focus_set:
            keep_mask.append(sc >= focus_thresh)
        else:
            keep_mask.append(sc >= strict_thresh)
    if not any(keep_mask):
        return (np.zeros((0, 4), dtype=np.float32),
                np.zeros((0,), dtype=np.float32), [])
    keep_idx = [i for i, k in enumerate(keep_mask) if k]
    boxes = boxes[keep_idx]
    scores = scores[keep_idx]
    cleaned = [cleaned[i] for i in keep_idx]

    # top-K (ưu tiên focus)
    if len(boxes) > cfg.MAX_BOXES:
        is_focus = torch.tensor([c in focus_set for c in cleaned])
        focus_idx = torch.nonzero(is_focus, as_tuple=False).flatten().tolist()
        non_idx = torch.nonzero(~is_focus, as_tuple=False).flatten().tolist()
        focus_sorted = sorted(focus_idx, key=lambda i: -scores[i].item())
        non_sorted = sorted(non_idx, key=lambda i: -scores[i].item())
        chosen = (focus_sorted + non_sorted)[:cfg.MAX_BOXES]
        chosen = sorted(chosen)
        boxes = boxes[chosen]
        scores = scores[chosen]
        cleaned = [cleaned[i] for i in chosen]

    return boxes.numpy().astype(np.float32), scores.numpy().astype(np.float32), cleaned


print('GroundingDINO helpers ready (multi-video + decord prefetch, padded text)')

In [ ]:
# CELL 6: ROI feature & class-text embedding helpers (single + multi)
# Keep detectron2 import stable
import os
import sys
from pathlib import Path

D2_DIR = Path('/content/detectron2')
os.chdir(D2_DIR)
if str(D2_DIR) not in sys.path:
    sys.path.insert(0, str(D2_DIR))

from detectron2.structures import ImageList, Boxes


def _frcnn_preprocess_frame(frame_rgb):
    bgr = frame_rgb[:, :, ::-1].copy()
    img = torch.from_numpy(bgr).permute(2, 0, 1).float().to(cfg.DEVICE)
    pixel_mean = torch.tensor(frcnn_cfg.MODEL.PIXEL_MEAN).view(-1, 1, 1).to(cfg.DEVICE)
    pixel_std = torch.tensor(frcnn_cfg.MODEL.PIXEL_STD).view(-1, 1, 1).to(cfg.DEVICE)
    img = (img - pixel_mean) / pixel_std
    images = ImageList.from_tensors([img], frcnn_model.backbone.size_divisibility)
    features = frcnn_model.backbone(images.tensor)
    return images, features


@torch.no_grad()
def extract_roi_features(frame_rgb, boxes_xyxy):
    if len(boxes_xyxy) == 0:
        return np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32)
    _, features = _frcnn_preprocess_frame(frame_rgb)
    boxes_tensor = torch.as_tensor(boxes_xyxy, device=cfg.DEVICE, dtype=torch.float32)
    boxes_d2 = Boxes(boxes_tensor)
    res4 = features['res4']
    roi_feats = frcnn_model.roi_heads.pooler([res4], [boxes_d2])
    res5_feats = frcnn_model.roi_heads.res5(roi_feats)
    feat_vec = torch.mean(res5_feats, dim=[2, 3])
    return feat_vec.float().cpu().numpy().astype(np.float32)


@torch.no_grad()
def extract_roi_features_multi(crop_jobs):
    """crop_jobs: list[(frame_rgb, boxes_xyxy_orig)] đồng thứ tự với records.

    Dùng FasterRCNN per-frame để lấy ROI feature cho từng frame.
    Trả về list np.ndarray (n_boxes_i, HIDDEN_DIM) đúng thứ tự crop_jobs.
    """
    out = []
    for frame_rgb, boxes in crop_jobs:
        if frame_rgb is None or len(boxes) == 0:
            out.append(np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32))
            continue
        out.append(extract_roi_features(frame_rgb, boxes))
    return out


_TEXT_CACHE = {}


@torch.no_grad()
def encode_class_texts(texts):
    """DeBERTa [CLS] embedding (768-d) cho list label string, có cache toàn cục."""
    out = np.zeros((len(texts), 768), dtype=np.float32)
    todo_idx, todo_txt = [], []
    for i, t in enumerate(texts):
        if t in _TEXT_CACHE:
            out[i] = _TEXT_CACHE[t]
        else:
            todo_idx.append(i); todo_txt.append(t)
    if todo_txt:
        for s in range(0, len(todo_txt), cfg.DEBERTA_BATCH):
            chunk = todo_txt[s:s + cfg.DEBERTA_BATCH]
            toks = deberta_tokenizer(chunk, padding=True, truncation=True,
                                     max_length=8, return_tensors='pt').to(cfg.DEVICE)
            with torch.autocast(device_type='cuda', dtype=cfg.DTYPE,
                                enabled=cfg.DEVICE == 'cuda'):
                emb = deberta_model(**toks).last_hidden_state[:, 0]
            emb = emb.float().cpu().numpy()
            for j, i in enumerate(todo_idx[s:s + cfg.DEBERTA_BATCH]):
                _TEXT_CACHE[chunk[j]] = emb[j]
                out[i] = emb[j]
    return out


print('ROI + class-text helpers ready (FasterRCNN, batched DeBERTa)')

In [ ]:
# CELL 7: Build video → path map and target list
# Tự auto-discover VIDEO_ROOT nếu path trong cfg không tồn tại
# hoặc rglob không tìm ra mp4 (Kaggle slug mount khác nhau giữa máy).

def _resolve_video_dir(preferred, candidates):
    """Return a Path that contains .mp4 files.
    1. nếu `preferred` có mp4 -> dùng
    2. scan các `candidates` (root dirs) tìm folder có nhiều mp4 nhất
    """
    pref = Path(preferred) if preferred else None
    if pref and pref.exists() and any(pref.rglob('*.mp4')):
        return pref
    best, best_n = None, 0
    for root in candidates:
        r = Path(root)
        if not r.exists():
            continue
        mp4s = list(r.rglob('*.mp4'))
        if len(mp4s) > best_n:
            best_n = len(mp4s)
            # lấy common parent của tất cả mp4 -> folder chứa video
            common = Path(os.path.commonpath([str(p) for p in mp4s]))
            best = common
    return best


ROOT_CANDIDATES = [
    cfg.VIDEO_ROOT,
    '/kaggle/input/raw-casual',
    '/kaggle/input/d/danielq07/raw-casual',
    '/kaggle/input/datasets/danielq07/raw-casual',
    '/content/data/raw-casual',
]

resolved_root = _resolve_video_dir(cfg.VIDEO_ROOT, ROOT_CANDIDATES)
print(f'🔎 resolved VIDEO_ROOT = {resolved_root}')
cfg.VIDEO_ROOT = str(resolved_root) if resolved_root else cfg.VIDEO_ROOT

root_map = {p.stem: p for p in Path(cfg.VIDEO_ROOT).rglob('*.mp4')} if resolved_root else {}
miss_map = {}  # disable missing-video pipeline
print(f'Videos found: root={len(root_map)}')

if cfg.SPLIT_LIST and Path(cfg.SPLIT_LIST).exists():
    with open(cfg.SPLIT_LIST) as f:
        target_ids = [ln.strip() for ln in f if ln.strip()]
else:
    target_ids = sorted(root_map)

# only keep ones with QA
qa_root = Path(cfg.QA_ROOT)
before_qa = len(target_ids)
target_ids = [v for v in target_ids if (qa_root / v).exists()]
print(f'After QA filter: {len(target_ids)} / {before_qa}  (QA_ROOT={qa_root})')

if cfg.TOTAL_NODES > 1:
    chunk = len(target_ids) // cfg.TOTAL_NODES
    s = cfg.NOTEBOOK_ID * chunk
    e = s + chunk if cfg.NOTEBOOK_ID < cfg.TOTAL_NODES - 1 else len(target_ids)
    target_ids = target_ids[s:e]

print(f'Will process {len(target_ids)} videos (node {cfg.NOTEBOOK_ID}/{cfg.TOTAL_NODES})')

In [ ]:
# CELL 8: 🧪 Smoke test — chạy thử 1 video (1 frame), dump pkl structure + render
import matplotlib.pyplot as plt
import matplotlib.patches as patches

TEST_VIDEO_ID = 'Ou7iBq0YBUQ_000102_000112'   # 🔁 đổi để test video khác
TEST_FRAMES = 1                               # smoke test: 1 frame cho nhẹ VRAM

def run_single_video(video_id, render=True, n_frames=TEST_FRAMES):
    vp = root_map.get(video_id) or miss_map.get(video_id)
    assert vp is not None, f'Không tìm thấy mp4 cho {video_id}'
    print(f'📼 mp4: {vp}')

    prompt, nouns, focus_nouns = build_prompt_for_video(video_id)
    assert prompt is not None, f'Không có QA cho {video_id} tại {cfg.QA_ROOT}'
    print(f'📝 Prompt ({len(nouns)} noun, focus={focus_nouns}): {prompt}')

    frames, indices, total = sample_frames(vp, n_frames)
    assert frames is not None, 'Không decode được video'
    h, w = next((f.shape[:2] for f in frames if f is not None), (0, 0))
    n_ok = sum(f is not None for f in frames)
    print(f'🎞  Frames: {n_ok}/{len(frames)} decoded | orig {w}x{h} | total={total}')

    # Gọi GDINO với threshold nới lỏng để không bỏ sót focus nouns
    relaxed_box  = max(0.05, cfg.BOX_THRESH  - cfg.FOCUS_THRESH_BONUS)
    relaxed_text = max(0.05, cfg.TEXT_THRESH - cfg.FOCUS_THRESH_BONUS)

    t0 = time.time()
    gd_results = run_gdino_batch(frames, prompt,
                                 box_thresh=relaxed_box,
                                 text_thresh=relaxed_text)
    print(f'🦖 GroundingDINO forward: {time.time() - t0:.2f}s '
          f'(box>={relaxed_box:.2f}, text>={relaxed_text:.2f})')

    per_frame_records = []
    unique_labels = OrderedDict()
    for fi, (frame_rgb, frame_idx, res) in enumerate(zip(frames, indices, gd_results)):
        if frame_rgb is None:
            per_frame_records.append({
                'frame_idx': int(frame_idx),
                'boxes_xyxy_orig': np.zeros((0, 4), dtype=np.float32),
                'scores': np.zeros((0,), dtype=np.float32),
                'labels_text': [],
                'roi_features': np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32),
            })
            continue
        fh, fw = frame_rgb.shape[:2]
        boxes, scores, labels = filter_boxes(res, fh, fw, nouns, focus_nouns)
        roi = extract_roi_features(frame_rgb, boxes)
        for lab in labels:
            unique_labels.setdefault(lab, None)
        per_frame_records.append({
            'frame_idx': int(frame_idx),
            'boxes_xyxy_orig': boxes,
            'scores': scores,
            'labels_text': labels,
            'roi_features': roi,
        })

    uniq_list = list(unique_labels.keys())
    if uniq_list:
        uniq_emb = encode_class_texts(uniq_list)
        emb_map = dict(zip(uniq_list, uniq_emb))
    else:
        emb_map = {}
    for rec in per_frame_records:
        if rec['labels_text']:
            rec['class_text_embedding'] = np.stack(
                [emb_map[l] for l in rec['labels_text']]
            ).astype(np.float32)
        else:
            rec['class_text_embedding'] = np.zeros((0, 768), dtype=np.float32)

    package = {
        'video_id': video_id,
        'orig_h': int(h), 'orig_w': int(w),
        'total_frames': int(total),
        'prompt': prompt,
        'nouns': nouns,
        'focus_nouns': focus_nouns,
        'frames': per_frame_records,
    }

    # ---- Dump cấu trúc pkl ----
    print('\n' + '=' * 80)
    print(f'📦 PKL STRUCTURE cho {video_id}')
    print('=' * 80)
    for k, v in package.items():
        if k == 'frames':
            print(f'  {k}: list[{len(v)}] of frame dict')
        elif isinstance(v, (list, tuple)):
            print(f'  {k}: {type(v).__name__}[{len(v)}] = {v}')
        else:
            print(f'  {k}: {type(v).__name__} = {v}')

    print(f'\n📑 Unique class labels trong video: {len(uniq_list)}')
    print('   ', uniq_list)

    total_boxes = 0
    for i, rec in enumerate(per_frame_records):
        n = len(rec['labels_text'])
        total_boxes += n
        tag = f'frame[{i:02d}] idx={rec["frame_idx"]:>4}  n={n:>2}'
        shapes = (f'boxes={rec["boxes_xyxy_orig"].shape}  '
                  f'scores={rec["scores"].shape}  '
                  f'roi={rec["roi_features"].shape}  '
                  f'cls_emb={rec["class_text_embedding"].shape}')
        labels_preview = rec['labels_text'][:6]
        scores_preview = [f'{s:.2f}' for s in rec['scores'][:6].tolist()]
        print(f'  {tag}  {shapes}')
        if n:
            print(f'        labels={labels_preview}  scores={scores_preview}')
    print(f'\n➡️  Tổng box: {total_boxes} | TB {total_boxes / max(len(per_frame_records),1):.2f}/frame')

    # ---- Render (1 frame) ----
    if render:
        focus_set = set(focus_nouns)
        fig, ax = plt.subplots(1, 1, figsize=(7, 5))
        rec = per_frame_records[0]
        frame_rgb = frames[0]
        if frame_rgb is None:
            ax.axis('off')
        else:
            ax.imshow(frame_rgb)
            for (x1, y1, x2, y2), lab, sc in zip(
                    rec['boxes_xyxy_orig'], rec['labels_text'], rec['scores']):
                is_focus = lab in focus_set
                edge = 'red' if is_focus else 'lime'
                ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                               lw=3 if is_focus else 2,
                                               edgecolor=edge, facecolor='none'))
                ax.text(x1, max(0, y1 - 4), f'{lab} {sc:.2f}{" ★" if is_focus else ""}',
                        fontsize=10, color='black',
                        bbox=dict(facecolor=edge, alpha=0.85, pad=2))
            ax.set_title(f"frame {rec['frame_idx']} ({len(rec['labels_text'])} boxes, "
                         f"red=focus)", fontsize=11)
            ax.axis('off')
        plt.suptitle(f'{video_id}', fontsize=11)
        plt.tight_layout()
        plt.show()

    return package


test_package = run_single_video(TEST_VIDEO_ID, render=True, n_frames=TEST_FRAMES)
print('\n✅ Smoke test done. Nếu OK, chạy cell sau để process toàn bộ.')


In [ ]:
# import gc
# import torch
# import ctypes

# # 1. Đóng các handle còn sót (nếu loop crash giữa chừng)
# try:
#     executor.shutdown(wait=False)
#     zf.close()
# except:
#     pass

# # 2. Xóa các biến chứa dữ liệu nặng trong namespace
# # Lưu ý: Ta dùng list các tên biến để tránh lỗi NameError nếu biến chưa được khởi tạo
# heavy_vars = [
#     'group_data', 'gd_results', 'crop_jobs', 'roi_outs',
#     'pending_records', 'inflight', 'video_items', 'emb_map'
# ]

# for var in heavy_vars:
#     if var in globals():
#         del globals()[var]

# # 3. Giải phóng bộ nhớ Python (System RAM)
# gc.collect()

# # 4. Giải phóng VRAM (GPU) - Cực kỳ quan trọng cho GDINO và DINOv3
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.cuda.ipc_collect() # Giải phóng bộ nhớ liên tiến trình nếu có
#     print(f"VRAM hiện tại: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

# # 5. Ép Linux trả lại RAM cho hệ thống (Malloc Trim)
# # Colab chạy trên Ubuntu, nên lệnh này rất hiệu quả để tránh crash "Your session crashed..."
# try:
#     libc = ctypes.CDLL("libc.so.6")
#     libc.malloc_trim(0)
# except Exception as e:
#     print(f"Malloc trim failed: {e}")

# print("✅ Đã dọn dẹp xong RAM và VRAM.")

In [ ]:
# CELL: Resume from Google Drive zip (mount + copy)
# Nếu bạn đã lưu zip lên Google Drive, cell này sẽ mount Drive và copy file zip
# vào `cfg.OUTPUT_DIR` để notebook có thể resume tiếp. Thay `DRIVE_ZIP_PATH`
# nếu cần (ví dụ: '/content/drive/MyDrive/gdino_backup/gdino_strict_v1.zip').


DRIVE_ZIP_PATH = '/content/drive/MyDrive/gdino_backup/gdino_strict_v1_backup_20260514_125552.zip'  # <-- chỉnh nếu cần

if IS_COLAB_DRIVE:
    print('Mounting Google Drive...')
    drive.mount('/content/drive')
    src = Path(DRIVE_ZIP_PATH)
    dst = cfg.OUTPUT_DIR / cfg.ZIP_NAME
    if src.exists():
        try:
            import shutil
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f'Copied {src} → {dst}')
        except Exception as e:
            print('Copy failed:', e)
    else:
        print(f'⚠️ Không thấy file trên Drive: {src} — nếu muốn resume, cập nhật DRIVE_ZIP_PATH')
else:
    print('Không phải Colab hoặc Drive không khả dụng — bỏ qua bước copy zip từ Drive.')

In [ ]:
# CELL 9: Main loop — prefetch decode + GDINO/DeBERTa batched, resume-safe
# Pipeline: ThreadPool decode video song song với GPU forward.
# Tối ưu: GDINO chạy theo group bằng run_gdino_multi để tận dụng VRAM tốt hơn.

from concurrent.futures import ThreadPoolExecutor

zip_path = cfg.OUTPUT_DIR / cfg.ZIP_NAME

# Resume: nếu zip đã có, giữ lại các pkl cũ, chỉ chạy những vid còn thiếu.
done_vids = set()
if zip_path.exists():
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf_old:
            done_vids = {Path(n).stem for n in zf_old.namelist() if n.endswith('.pkl')}
        print(f'↻ Resume: {len(done_vids)} video đã có trong zip cũ, sẽ bổ sung phần còn lại.')
    except Exception as e:
        print(f'⚠️  Không đọc được zip cũ ({e}), sẽ ghi mới.')
        done_vids = set()

print('Writing to:', zip_path)
# mode='a' nếu file tồn tại, ngược lại 'w'
zf_mode = 'a' if zip_path.exists() and done_vids else 'w'
zf = zipfile.ZipFile(zip_path, zf_mode, compression=zipfile.ZIP_DEFLATED, allowZip64=True)

pending_ids = [v for v in target_ids if v not in done_vids]
print(f'Pending: {len(pending_ids)} / {len(target_ids)} videos')

relaxed_box  = max(0.05, cfg.BOX_THRESH  - cfg.FOCUS_THRESH_BONUS)
relaxed_text = max(0.05, cfg.TEXT_THRESH - cfg.FOCUS_THRESH_BONUS)


def _prepare_video(vid):
    vp = root_map.get(vid) or miss_map.get(vid)
    if vp is None:
        return {'vid': vid, 'status': 'no_video'}
    prompt, nouns, focus_nouns = build_prompt_for_video(vid)
    if prompt is None:
        return {'vid': vid, 'status': 'no_video'}
    frames, indices, total = sample_frames(vp, cfg.N_FRAMES)
    if frames is None:
        return {'vid': vid, 'status': 'fail'}
    h, w = next((f.shape[:2] for f in frames if f is not None), (0, 0))
    if h == 0:
        return {'vid': vid, 'status': 'fail'}
    return {
        'vid': vid, 'status': 'ok',
        'frames': frames, 'indices': indices, 'total': total,
        'prompt': prompt, 'nouns': nouns, 'focus_nouns': focus_nouns,
        'orig_h': int(h), 'orig_w': int(w),
    }


def _process_group(group_data):
    stats = {'ok': 0, 'fail': 0, 'no_video': 0}
    valid = []
    for d in group_data:
        if d is None:
            continue
        if d['status'] == 'no_video':
            stats['no_video'] += 1
        elif d['status'] == 'fail':
            stats['fail'] += 1
        elif d['status'] == 'ok':
            valid.append(d)
    if not valid:
        return stats

    # ---- GDINO forward (batch theo group để tăng GPU utilization) ----
    gd_results = {}
    try:
        video_items = [(d['vid'], d['frames'], d['prompt']) for d in valid]
        gd_results = run_gdino_multi(
            video_items,
            box_thresh=relaxed_box,
            text_thresh=relaxed_text,
        )
    except torch.cuda.OutOfMemoryError as e:
        print(f'  ⚠️ GDINO group OOM, fallback per-video: {e}')
        torch.cuda.empty_cache()
        for d in valid:
            try:
                gd_results[d['vid']] = run_gdino_batch(
                    d['frames'], d['prompt'],
                    box_thresh=relaxed_box, text_thresh=relaxed_text,
                )
            except torch.cuda.OutOfMemoryError as e2:
                print(f'  ❌ OOM {d["vid"]}: {e2}')
                torch.cuda.empty_cache()
                gd_results[d['vid']] = [None] * len(d['frames'])
            except Exception as e2:
                print(f'  ❌ {d["vid"]}: {e2}')
                gd_results[d['vid']] = [None] * len(d['frames'])
    except Exception as e:
        print(f'  ❌ GDINO group error: {e}')
        for d in valid:
            gd_results[d['vid']] = [None] * len(d['frames'])

    # ---- Filter boxes per frame ----
    crop_jobs = []
    job_meta = []
    pending_records = []
    for v_i, d in enumerate(valid):
        per_frame = []
        res_v = gd_results.get(d['vid'], [None] * len(d['frames']))
        for f_i, (frame_rgb, frame_idx, res) in enumerate(
                zip(d['frames'], d['indices'], res_v)):
            if frame_rgb is None:
                per_frame.append({
                    'frame_idx': int(frame_idx),
                    'boxes_xyxy_orig': np.zeros((0, 4), dtype=np.float32),
                    'scores': np.zeros((0,), dtype=np.float32),
                    'labels_text': [],
                    'roi_features': np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32),
                })
                continue
            fh, fw = frame_rgb.shape[:2]
            boxes, scores, labels = filter_boxes(res, fh, fw, d['nouns'], d['focus_nouns'])
            per_frame.append({
                'frame_idx': int(frame_idx),
                'boxes_xyxy_orig': boxes,
                'scores': scores,
                'labels_text': labels,
                'roi_features': None,
            })
            crop_jobs.append((frame_rgb, boxes))
            job_meta.append((v_i, f_i))
        pending_records.append(per_frame)

    # ---- ROI forward (gom toàn group) ----
    if crop_jobs:
        try:
            roi_outs = extract_roi_features_multi(crop_jobs)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            # fallback per-frame
            roi_outs = []
            for fr, bx in crop_jobs:
                try:
                    roi_outs.append(extract_roi_features(fr, bx))
                except Exception:
                    roi_outs.append(np.zeros((len(bx), cfg.HIDDEN_DIM), dtype=np.float32))
    else:
        roi_outs = []
    for (v_i, f_i), feats in zip(job_meta, roi_outs):
        pending_records[v_i][f_i]['roi_features'] = feats
    for recs in pending_records:
        for rec in recs:
            if rec['roi_features'] is None:
                rec['roi_features'] = np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32)

    # ---- DeBERTa class-text embed ----
    group_unique = []
    seen_in_group = set()
    for recs in pending_records:
        for rec in recs:
            for lab in rec['labels_text']:
                if lab not in seen_in_group:
                    seen_in_group.add(lab)
                    group_unique.append(lab)
    if group_unique:
        emb = encode_class_texts(group_unique)
        emb_map = dict(zip(group_unique, emb))
    else:
        emb_map = {}
    for recs in pending_records:
        for rec in recs:
            if rec['labels_text']:
                rec['class_text_embedding'] = np.stack(
                    [emb_map[l] for l in rec['labels_text']]
                ).astype(np.float32)
            else:
                rec['class_text_embedding'] = np.zeros((0, 768), dtype=np.float32)

    # ---- Ghi zip ----
    for d, recs in zip(valid, pending_records):
        package = {
            'video_id': d['vid'],
            'orig_h': d['orig_h'], 'orig_w': d['orig_w'],
            'total_frames': int(d['total']),
            'prompt': d['prompt'],
            'nouns': d['nouns'],
            'focus_nouns': d['focus_nouns'],
            'frames': recs,
        }
        zf.writestr(f"{d['vid']}.pkl",
                    pickle.dumps(package, protocol=pickle.HIGHEST_PROTOCOL))
        stats['ok'] += 1
    return stats


groups = [pending_ids[i:i + cfg.VIDEO_BATCH]
          for i in range(0, len(pending_ids), cfg.VIDEO_BATCH)]

executor = ThreadPoolExecutor(max_workers=cfg.PREFETCH_WORKERS)


def _decode_group(group_ids):
    return [_prepare_video(v) for v in group_ids]


inflight = []
prefetch_n = max(1, cfg.PREFETCH_WORKERS)
next_idx = 0
while next_idx < len(groups) and len(inflight) < prefetch_n:
    inflight.append(executor.submit(_decode_group, groups[next_idx]))
    next_idx += 1

ok_total, fail_total, novid_total = 0, 0, 0
t0 = time.time()
pbar = tqdm(total=len(pending_ids), desc='Videos')
g_processed = 0
while inflight:
    fut = inflight.pop(0)
    group_data = fut.result()
    if next_idx < len(groups):
        inflight.append(executor.submit(_decode_group, groups[next_idx]))
        next_idx += 1
    try:
        stats = _process_group(group_data)
    except Exception as e:
        print(f'❌ group error: {e}')
        stats = {'ok': 0, 'fail': len(group_data), 'no_video': 0}
    ok_total += stats['ok']; fail_total += stats['fail']; novid_total += stats['no_video']
    pbar.update(len(group_data))
    g_processed += 1
    if cfg.EMPTY_CACHE_EVERY and g_processed % cfg.EMPTY_CACHE_EVERY == 0:
        torch.cuda.empty_cache()
    if g_processed % 50 == 0:
        elapsed = time.time() - t0
        rate = (ok_total + fail_total + novid_total) / max(elapsed, 1e-6)
        pbar.set_postfix(ok=ok_total, fail=fail_total, vps=f'{rate:.2f}')

pbar.close()
executor.shutdown(wait=True)
zf.close()

elapsed = time.time() - t0
size_gb = zip_path.stat().st_size / 1e9
done = ok_total + fail_total + novid_total
print(f'\n✅ Done. ok={ok_total} fail={fail_total} no_video={novid_total}')
print(f'   Zip: {zip_path}  ({size_gb:.2f} GB)')
print(f'   Elapsed: {elapsed/3600:.2f} h ({elapsed/max(ok_total,1):.2f} s/video, '
      f'throughput {done/max(elapsed,1e-6):.2f} videos/s)')

In [ ]:
# CELL: Backup zip hiện tại sang Google Drive

from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path
from datetime import datetime

# File zip đang được tạo
src = Path('/content/gdino_strict/gdino_strict_v1.zip')

# Thư mục lưu backup trên Google Drive
backup_dir = Path('/content/drive/MyDrive/gdino_backup')
backup_dir.mkdir(parents=True, exist_ok=True)

# Tạo tên file backup có timestamp để không bị ghi đè
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dst = backup_dir / f'gdino_strict_v1_backup_{timestamp}.zip'

if src.exists():
    shutil.copy2(src, dst)
    print(f'✅ Đã copy backup sang Google Drive:')
    print(dst)
    print(f'Dung lượng: {dst.stat().st_size / 1e9:.2f} GB')
else:
    print(f'❌ Không tìm thấy file zip: {src}')

In [ ]:
# CELL 9: Upload to HuggingFace (path = gdinofasterrcnn/<PART_ZIP>)
if cfg.HF_TOKEN and cfg.HF_REPO_ID:
    login(token=cfg.HF_TOKEN, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(cfg.HF_REPO_ID, repo_type='dataset', exist_ok=True, token=cfg.HF_TOKEN)
    print(f'⬆️  Uploading {zip_path.name} → {cfg.HF_REPO_ID}/{cfg.HF_OUT_PATH} ...')
    api.upload_file(
        path_or_fileobj=str(zip_path),
        path_in_repo=cfg.HF_OUT_PATH,
        repo_id=cfg.HF_REPO_ID,
        repo_type='dataset',
        token=cfg.HF_TOKEN,
    )
    print(f'✅ Upload complete: https://huggingface.co/datasets/{cfg.HF_REPO_ID}/blob/main/{cfg.HF_OUT_PATH}')
else:
    print('⚠️ HF credentials not set — skipping upload.')

In [ ]:
# CELL 10: Verification — load random pkl, assert schema, render one frame
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = [n for n in zf.namelist() if n.endswith('.pkl')]
    assert names, 'No pkls in zip!'
    sample_name = random.choice(names)
    pkg = pickle.loads(zf.read(sample_name))

print('Sample:', sample_name)
print('  video_id:', pkg['video_id'])
print('  orig_hxw:', pkg['orig_h'], 'x', pkg['orig_w'])
print('  prompt:', pkg['prompt'][:160], '...')
print('  num_frames:', len(pkg['frames']))
for i, fr in enumerate(pkg['frames'][:3]):
    print(f"  frame[{i}] idx={fr['frame_idx']} boxes={fr['boxes_xyxy_orig'].shape} "
          f"roi={fr['roi_features'].shape} cls_emb={fr['class_text_embedding'].shape} "
          f"labels={fr['labels_text'][:5]}")

# render: open same video, decode 1 frame, draw boxes
vp = root_map.get(pkg['video_id']) or miss_map.get(pkg['video_id'])
if vp is not None:
    cap = cv2.VideoCapture(str(vp))
    target_fr = pkg['frames'][len(pkg['frames']) // 2]
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(target_fr['frame_idx']))
    ok, bgr = cap.read(); cap.release()
    if ok:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(rgb)
        for (x1, y1, x2, y2), lab, sc in zip(
                target_fr['boxes_xyxy_orig'], target_fr['labels_text'], target_fr['scores']):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           lw=2, edgecolor='lime', facecolor='none'))
            ax.text(x1, max(0, y1 - 4), f'{lab} {sc:.2f}',
                    color='black', fontsize=10,
                    bbox=dict(facecolor='lime', alpha=0.8, pad=2))
        ax.set_title(f"{pkg['video_id']}  frame {target_fr['frame_idx']}  ({len(target_fr['labels_text'])} boxes)")
        ax.axis('off')
        plt.tight_layout(); plt.show()


In [ ]:
from google.colab import runtime
runtime.unassign()